# CelebA Face Generator — Simple TensorFlow Diffusion
**A small, from-scratch diffusion lesson. All model code is in this notebook.**

Run **Runtime → Change runtime type → GPU**, then run the five code cells in order.

**Load 5,000 small faces → add noise → train a noise predictor → generate new faces.**

We use center-cropped **32 × 32 RGB** images, a small U-Net, and no text prompts. This is a teaching model, not a pretrained photorealistic face generator. More steps can improve results; there is no fixed-time or quality guarantee. Training duration is not benchmarked.

## 1. Load a small CelebA subset
Read only the first **5,000 examples** from a pinned public CelebA mirror using streaming. Crop around the center and resize each face to **32 × 32**. Cache the small array locally so rerunning this cell avoids repeating preprocessing.

Streaming avoids intentionally downloading the entire dataset, although the service may transfer larger Parquet blocks than the exact selected images. This first-N subset is a simple teaching sample, not a balanced or representative benchmark.

CelebA images keep their original dataset terms; this repository does not redistribute them. See the [official CelebA page](https://mmlab.ie.cuhk.edu.hk/projects/CelebA.html).

In [ ]:
# Install only the dataset reader. Colab already includes TensorFlow, NumPy and Pillow.
import importlib.util, subprocess, sys
if importlib.util.find_spec('datasets') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'datasets>=3,<5'])

from pathlib import Path
from io import BytesIO
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from datasets import load_dataset, Image as DatasetImage
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.keras.utils.set_random_seed(42)
NUM_IMAGES = 5000  # A small subset instead of the full CelebA dataset.
SIZE = 32         # Smaller images make both the model and training simpler.
cache = Path(f'celeba_{NUM_IMAGES}_{SIZE}.npz')

if cache.exists():
    with np.load(cache) as saved:
        images = saved['images']
else:
    stream = load_dataset('nielsr/CelebA-faces', split='train', streaming=True,
        revision='db26feecbaed40316ee4c80ec0e72211a2f4afd6')
    # Read image bytes directly; Pillow handles conversion and resizing.
    stream = stream.cast_column('image', DatasetImage(decode=False))
    pictures = []
    for row in stream.take(NUM_IMAGES):
        with Image.open(BytesIO(row['image']['bytes'])) as original:
            face = ImageOps.fit(original.convert('RGB'), (SIZE, SIZE),
                method=Image.Resampling.LANCZOS, bleed=0.1)  # Center crop + resize.
            pictures.append(np.asarray(face))
        if len(pictures) % 1000 == 0:
            print('Faces prepared:', len(pictures))
    images = np.stack(pictures).astype('uint8')
    np.savez_compressed(cache, images=images)

# Store uint8 pixels to save memory; normalize only the selected training batch.
image_bank = tf.constant(images)
print('Images:', images.shape, '| GPU:', bool(tf.config.list_physical_devices('GPU')))

def show_images(pictures, title):
    pictures = np.clip((np.asarray(pictures) + 1) / 2, 0, 1)  # [-1, 1] → [0, 1].
    fig, axes = plt.subplots(1, len(pictures), figsize=(2 * len(pictures), 2.5), squeeze=False)
    for ax, picture in zip(axes[0], pictures):
        ax.imshow(picture, interpolation='nearest')
        ax.axis('off')
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

show_images(images[:6].astype('float32') / 127.5 - 1, 'Actual CelebA training examples')

## 2. Add noise
\[
x_t=\sqrt{\bar\alpha_t}\,x_0+\sqrt{1-\bar\alpha_t}\,\epsilon.
\]

$x_0$ is the clean image scaled to −1 to 1. We sample the Gaussian noise $\epsilon$, so we know the answer the network should predict. $\bar\alpha_t$ is the fraction of signal variance remaining. It decreases as the noise level increases.

We use a cosine schedule with **200 noisy levels**. Code indices 0–199 represent those levels; the clean image is displayed separately.

In [ ]:
T = 200  # Number of noise levels.
curve = np.cos((np.linspace(0, 1, T + 1) + 0.008) / 1.008 * np.pi / 2) ** 2
curve = curve / curve[0]
beta = np.clip(1 - curve[1:] / curve[:-1], 0, 0.999)
alpha_bar = tf.constant(np.cumprod(1 - beta), dtype=tf.float32)

def add_noise(clean, t, noise):
    # One noise level per image, broadcast over height, width and RGB channels.
    a = tf.reshape(tf.gather(alpha_bar, t), [-1, 1, 1, 1])
    return tf.sqrt(a) * clean + tf.sqrt(1 - a) * noise

clean = tf.cast(image_bank[:1], tf.float32) / 127.5 - 1
noise = tf.random.normal(tf.shape(clean))
levels = [0, 40, 80, 120, 160, 199]
frames = [clean] + [add_noise(clean, [t], noise) for t in levels]
show_images(tf.concat(frames, axis=0), 'Clean face → increasing noise')

## 3. Build a small U-Net
The inputs are a **noisy face** and its **noise level**. The output is the predicted noise, with the same image shape. No face labels, text encoder, pretrained weights, or prompts are needed.

The encoder reduces spatial resolution **32 → 16 → 8**. The decoder expands it back to 32. Skip connections carry details across the network. The time condition enters each convolution block. Layer normalization helps keep feature values stable.

In [ ]:
noisy_input = keras.Input((SIZE, SIZE, 3), name='noisy_face')
time_input = keras.Input((1,), name='noise_level')  # Normalize level to [0, 1].
time_features = layers.Dense(128, activation='swish')(time_input)
time_features = layers.Dense(128, activation='swish')(time_features)

def conv_block(x, channels):
    x = layers.Conv2D(channels, 3, padding='same')(x)
    x = layers.LayerNormalization()(x)
    # Broadcast the noise-level information to every spatial position.
    hint = layers.Reshape((1, 1, channels))(layers.Dense(channels)(time_features))
    x = layers.Activation('swish')(layers.Add()([x, hint]))
    return layers.Conv2D(channels, 3, padding='same', activation='swish')(x)

a = conv_block(noisy_input, 32)                    # 32 × 32
b = conv_block(layers.AveragePooling2D(2)(a), 64)   # 16 × 16
middle = conv_block(layers.AveragePooling2D(2)(b), 96)  # 8 × 8
x = conv_block(layers.Concatenate()([layers.UpSampling2D()(middle), b]), 64)
x = conv_block(layers.Concatenate()([layers.UpSampling2D()(x), a]), 32)
predicted_noise = layers.Conv2D(3, 3, padding='same')(x)  # Linear output.
model = keras.Model([noisy_input, time_input], predicted_noise)
print('Trainable parameters:', model.count_params())

## 4. Train the noise predictor
Pick a batch of clean faces, sample random noise levels, add known noise, and train the network to predict it:
\[
\mathrm{loss}=\operatorname{mean}[(\epsilon-\hat\epsilon)^2].
\]

**STEPS** means additional training batches. Rerunning this cell continues the same model and optimizer. Rerunning Section 3 creates a fresh model. Loss measures noise prediction; it does not directly measure face realism.

In [ ]:
STEPS = 5000       # Additional training batches. Increase if faces are still unclear.
BATCH_SIZE = 64    # Reduce to 32 if GPU memory is limited.

if 'optimizer_model' not in globals() or optimizer_model is not model:
    optimizer = keras.optimizers.Adam(learning_rate=0.0003, clipnorm=1.0)
    optimizer_model = model

@tf.function  # Compile this step for faster execution.
def train_step():
    ids = tf.random.uniform([BATCH_SIZE], 0, len(images), dtype=tf.int32)
    clean = tf.cast(tf.gather(image_bank, ids), tf.float32) / 127.5 - 1
    t = tf.random.uniform([BATCH_SIZE], 0, T, dtype=tf.int32)
    noise = tf.random.normal(tf.shape(clean))
    noisy = add_noise(clean, t, noise)
    time = tf.cast(t[:, None], tf.float32) / (T - 1)
    with tf.GradientTape() as tape:
        prediction = model([noisy, time], training=True)
        loss = tf.reduce_mean(tf.square(noise - prediction))
    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    return loss

losses = []
for step in range(STEPS):
    losses.append(float(train_step()))
    if (step + 1) % 100 == 0:
        print(f'Step {step + 1}/{STEPS} | noise MSE: {np.mean(losses[-100:]):.4f}')

plt.plot(losses)
plt.xlabel('Training step'); plt.ylabel('Noise prediction MSE')
plt.title('Learning to predict noise'); plt.show()

## 5. Generate faces from fresh noise
Start with Gaussian noise. At each selected level, predict the noise, estimate a clean face, and move to a lower noise level:
\[
\hat x_0=\frac{x_t-\sqrt{1-\bar\alpha_t}\,\hat\epsilon}{\sqrt{\bar\alpha_t}},
\qquad
x_s=\sqrt{\bar\alpha_s}\,\hat x_0+\sqrt{1-\bar\alpha_s}\,\hat\epsilon,\quad s<t.
\]

This is deterministic **DDIM sampling** with 60 evaluations of the trained network. The initial seed provides randomness; no fresh noise is added at later steps. Change **SEED** to generate another batch. Keep the seed fixed when comparing results before and after more training.

The first row shows generated faces. The second shows selected noisy states for the first face. Nothing is drawn by a procedural face renderer.

In [ ]:
SEED = 42  #@param {type:"integer"}
NUMBER_OF_FACES = 6
x = tf.random.stateless_normal([NUMBER_OF_FACES, SIZE, SIZE, 3], seed=[SEED, 0])
times = np.linspace(T - 1, 0, 60, dtype=int)
frames = [x[:1]]

@tf.function
def predict_noise(x, time):
    return model([x, time], training=False)

for i, t in enumerate(times):
    time = tf.fill([NUMBER_OF_FACES, 1], tf.cast(t, tf.float32) / (T - 1))
    noise = predict_noise(x, time)
    a = alpha_bar[t]
    clean = (x - tf.sqrt(1 - a) * noise) / tf.sqrt(a)
    clean = tf.clip_by_value(clean, -1, 1)  # Valid pixel range.
    # Recompute noise after clipping to keep the two terms consistent with x.
    noise = (x - tf.sqrt(a) * clean) / tf.sqrt(1 - a)
    previous_a = alpha_bar[times[i + 1]] if i + 1 < len(times) else tf.constant(1.0)
    x = tf.sqrt(previous_a) * clean + tf.sqrt(1 - previous_a) * noise
    if (i + 1) % 12 == 0:
        frames.append(x[:1])

show_images(x, 'Generated faces — neural-network output')
show_images(tf.concat(frames, axis=0), 'Reverse diffusion: noise → face')

### Try it
- Change the seed and generate again.
- Rerun Section 4 for more training, then generate with the same seed.
- Inspect whether eyes, nose, mouth, and overall face shape improve. Do not judge progress from loss alone.

**Keep it simple:** This notebook intentionally has no prompts, attributes, guidance controls, model checkpoints, EMA, widgets, or validation dashboard. Restarting the runtime loses the trained model and may also lose the local data cache. It is designed to explain the mechanism rather than produce sharp, identity-controlled portraits.

References: [DDPM](https://arxiv.org/abs/2006.11239), [DDIM](https://arxiv.org/abs/2010.02502), [CelebA](https://mmlab.ie.cuhk.edu.hk/projects/CelebA.html).